# 03 — A2A

Drive the provider's A2A executor in-process. We fabricate a request context and an EventQueue, no port, no httpx — pure Python.

## Setup

In [ ]:
import sys, pathlib
_ROOT = pathlib.Path.cwd().resolve()
if (_ROOT / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT))
elif (_ROOT.parent / 'shared').is_dir():
    sys.path.insert(0, str(_ROOT.parent))


In [ ]:
from provider.agent_executor import BandwidthProviderExecutor
from provider.mcp_server import build_mcp_server
from shared.config import Config
from a2a.types import Message, Part
from google.protobuf.json_format import MessageToDict, ParseDict
from google.protobuf.struct_pb2 import Struct, Value
from unittest.mock import MagicMock

PROVIDER = '0x59c6995e998f97a5a0044966f0945389dc9e86dae88c7a8412f4603b6b78690d'
cfg = Config(provider_private_key=PROVIDER, sdn_mock=True)
mcp, _ = build_mcp_server(cfg)
executor = BandwidthProviderExecutor(mcp)
_components = mcp._local_provider._components
_tools = {v.name: v for k, v in _components.items() if k.startswith('tool:')}
print('executor ready, mcp has', len(_tools), 'tools')

## Build helpers

In [ ]:
class FakeQueue:
    def __init__(self):
        self.events = []
    async def enqueue_event(self, event):
        self.events.append(event)

def data_part(d: dict) -> Part:
    s = Struct(); ParseDict(d, s)
    return Part(data=Value(struct_value=s), media_type='application/json')

def make_context(payload: dict) -> MagicMock:
    msg = Message(message_id='m1', parts=[data_part(payload)])
    ctx = MagicMock()
    ctx.message = msg
    ctx.task_id = 'task-1'
    ctx.context_id = 'ctx-1'
    return ctx

def payload_of(event):
    return MessageToDict(event.artifact.parts[0].data,
                         preserving_proto_field_name=True)

## Run — three A2A actions

In [ ]:
import asyncio

async def call(payload):
    q = FakeQueue()
    await executor.execute(make_context(payload), q)
    return [payload_of(e) for e in q.events
            if hasattr(e, 'artifact')]

print('catalog:'); [print('  ', x) for x in await call({'action': 'get_catalog'})]
print('\nquote:');   [print('  ', x) for x in await call({
    'action': 'request_quote', 'package_id': 'small',
    'consumer_address': '0x000000000000000000000000000000000000dEaD'})]

## Inspect — the agent card the consumer would discover

In [ ]:
from provider.agent_card import build_provider_agent_card
card = build_provider_agent_card(cfg)
print('Skills advertised:')
for s in card.skills:
    print(' -', s.id, ':', s.name)